# Mini Project 1 · Group A
## Detection and Analysis of Performance Loss in a PLC-Controlled Production System Using Unsupervised Learning
### Stage 4: Unsupervised Anomaly Detection via 40-Forest Seed-Averaged Isolation Forest

**Lead Author:** Naga Sai Priyanka Truvidhula  
**Team Members:** Neeraj Bhuma, Lakshmi Narayana Reddy Mandi, Naga Sai Priyanka Truvidhula  
**Supervisor:** Prof. Felix Georg Müller  
**Institution:** Hochschule Albstadt-Sigmaringen (November 2026)

In [1]:
import numpy as np  # numerical tools
import pandas as pd  # tables
from sklearn.preprocessing import MinMaxScaler  # scaling to 0-1
from pathlib import Path
import os

# Dynamically locate project root, data file, and output directory
def get_project_dirs():
    current = Path.cwd().resolve()
    
    # 1. Output directory selection
    if (current / "02_Data_and_Outputs" / "Generated_Outputs").exists():
        out_dir = current / "02_Data_and_Outputs" / "Generated_Outputs"
    elif (current.parent / "02_Data_and_Outputs" / "Generated_Outputs").exists():
        out_dir = current.parent / "02_Data_and_Outputs" / "Generated_Outputs"
    elif (current / "Generated_Outputs").exists():
        out_dir = current / "Generated_Outputs"
    else:
        out_dir = current
    out_dir.mkdir(parents=True, exist_ok=True)
    
    # 2. Find merged_85ms.csv dynamically across project tree
    cands = [
        current / "merged_85ms.csv",
        out_dir / "merged_85ms.csv",
        current / "02_Data_and_Outputs" / "Generated_Outputs" / "merged_85ms.csv",
        current.parent / "02_Data_and_Outputs" / "Generated_Outputs" / "merged_85ms.csv",
        current.parent / "01_Neeraj_Bhuma_Data_Preparation_and_Synchronization" / "merged_85ms.csv",
        current / "01_Neeraj_Bhuma_Data_Preparation_and_Synchronization" / "merged_85ms.csv",
    ]
    csv_path = next((p for p in cands if p.exists()), None)
    if csv_path is None:
        for r in [current] + list(current.parents):
            matches = [p for p in r.glob("**/merged_85ms.csv") if "_Archive" not in p.parts and ".ipynb_checkpoints" not in p.parts]
            if matches:
                csv_path = matches[0]
                break
                
    if csv_path is None or not csv_path.exists():
        raise FileNotFoundError("Could not locate merged_85ms.csv across project directory tree.")
        
    return current, out_dir, csv_path

PROJECT_ROOT, OUT_DIR, CSV_PATH = get_project_dirs()
print(f"Loading merged data from: {CSV_PATH}")
print(f"Output directory set to:  {OUT_DIR}")
data = pd.read_csv(CSV_PATH)  # read the table

HELPERS = ['utc_time', 'timestamp_beckhoff', 'beck_time', 'beck_age_ms',
           'timestamp_siemens', 'sie_time', 'sie_age_ms', 't_s',
           'unix_ms', 'ms', 'BECK_src_ms', 'SIE_src_ms',
           'BECK_age_ms', 'SIE_age_ms']  # timing and merge columns
signals = [c for c in data.columns if c not in HELPERS]  # process signals

# Same audited exclusions as the approved feature selection:
# 40 signals constant in this recording + 3 exact Beckhoff copies.
CONSTANT_40 = [
    'VG.I3_ref_switch_rotate', 'WH.I4_Ref_swtich_vertical',
    'WH.A1_trail_sensor_lower', 'WH.A2_trail_sensor_upper',
    'VG.safeCurtainLong', 'Pallets.Conveyor_belt_lift',
    'VG.atHome', 'VG.error_robot', 'M34.4', 'M38.0', 'M38.1',
    'C0', 'EW288', 'C1', 'C2', 'M2.0', 'M2.1', 'M2.2', 'M2.3',
    'M3.0', 'M3.1', 'C4', 'M4.0', 'M5.7', 'M7.0', 'A6.4',
    'A6.5', 'M24.0', 'M24.1', 'A201.1', 'M24.2', 'EW258',
    'EW256', 'M24.5', 'M0.0', 'E6.0', 'E6.1', 'E200.1',
    'E1.7', 'E200.2',
]
COMM_COPIES = {'A200.4': 'A0.2', 'A200.5': 'A0.3', 'A200.6': 'A0.4'}  # copy

bad = [c for c in CONSTANT_40  # listed constants that are not constant
       if c not in data.columns or data[c].nunique(dropna=True) != 1]
bad += [copy for copy, original in COMM_COPIES.items()  # copies not equal
        if copy not in data.columns or original not in data.columns
        or not data[copy].equals(data[original])]
if bad:  # stop if the audit no longer holds
    raise ValueError(
        f'Audit conditions changed, review before excluding: {bad}')
if data[signals].isna().any().any():  # stop if a signal has gaps
    raise ValueError('Missing signal values found; review before proceeding.')

excluded = set(CONSTANT_40) | set(COMM_COPIES)  # 43 excluded names
features = [c for c in signals if c not in excluded]  # kept signals
X = data[features].copy()  # feature matrix, a copy of the source columns
if len(features) != 152:  # expected number of features
    raise ValueError(f'Expected 152 features, got {len(features)}')

# Same audited one-hot encoding: 7 code signals become indicator columns.
CODES = {  # code signals and the values seen in the audit
    'MW22': {0, 1, 2, 3},
    'AW256': {10, 4353, 13059, 40970, 43530},
    'MW26': {16384, 32768, 34816, 36864, 40960, 43008, 45056},
    'MD28': {33554435, 33554451},
    'MW32': {34819, 59395},
    'WH.desiredPalletX': {910, 1620, 1640, 2330},
    'WH.desiredPalletY': {110, 540, 1030},
}
for column, expected in CODES.items():  # check each code signal
    observed = set(X[column].dropna().unique())  # values in the data
    if X[column].isna().any() or observed != expected:
        raise ValueError(f'{column}: values differ from the audit: {observed}')
indicators = pd.get_dummies(X[list(CODES)].astype(str),  # 0/1 per value
                            prefix=list(CODES), dtype=int)
X_prepared = pd.concat([X.drop(columns=CODES), indicators], axis=1)  # 172
if X_prepared.shape != (len(X), 172):  # expected shape
    raise ValueError(f'Expected (rows, 172), got {X_prepared.shape}')

utc = pd.to_datetime(data['utc_time'], utc=True, format='ISO8601')  # times
if utc.isna().any() or not utc.is_monotonic_increasing:  # valid, ordered?
    raise ValueError('Source timestamps must be valid and ordered.')

# Scaling: min-max on the 5 numeric counters/sensors only.
NUMERIC = ['MW16', 'MW12', 'MW14', 'MW18', 'MW20']  # numeric signals
if not set(NUMERIC).issubset(X_prepared.columns):  # all 5 present?
    raise ValueError('Numeric columns missing from prepared matrix.')
scaler = MinMaxScaler()  # 0-1 scaler
X_scaled = X_prepared.copy()  # copy to hold the scaled values
X_scaled[NUMERIC] = scaler.fit_transform(X_prepared[NUMERIC])  # scale
print(f'Prepared feature matrix: {X_scaled.shape}')


Loading merged data from: /Users/lakshminarayanareddymandu/Downloads/B4 Project/02_Data_and_Outputs/Generated_Outputs/merged_85ms.csv
Source data: (5945, 202) (file unchanged)
Excluded from model matrix: 43 (40 constant, 3 identical copies)
X_prepared: (5945, 172) after one-hot of 7 code signals
X_minmax: (5945, 172) - only the 5 numeric columns scaled
Same audited preprocessing as before; no rows dropped, no model fitted.


**What:**
- Load the saved merged table merged_85ms.csv and rebuild the feature matrix with the same audited rules as in the feature selection notebook: 40 constant signals and 3 exact communication copies excluded (43), leaving 152 features
- One-hot encode the 7 code signals (MW22, AW256, MW26, MD28, MW32, WH.desiredPalletX, WH.desiredPalletY), giving X_prepared with 172 columns
- Min-max scale only the 5 numeric columns (MW16, MW12, MW14, MW18, MW20) to 0-1, giving X_minmax

**Why:**
- The cell loads its own data, so the Isolation Forest part runs without the earlier notebooks
- The checks stop the cell if the audit, the code values, the timestamps or the shape differ from what we approved
- The source file is not changed

**Output:**
- Source data: (5945, 202)
- Excluded from the model matrix: 43 (40 constant, 3 identical copies)
- X_prepared: (5945, 172) after one-hot of the 7 code signals
- X_minmax: (5945, 172), only the 5 numeric columns scaled; the other 167 are 0/1
- No rows dropped, no model fitted

**Next:** test how much the Isolation Forest result depends on its settings and on the random seed.

In [2]:
# IF sensitivity check: which setting changes the output, and by how much?
# Needs X_minmax and utc from the complete pipeline cell (same session).
import numpy as np  # numerical tools
import pandas as pd  # tables
from sklearn.ensemble import IsolationForest  # the model

values = X_minmax.to_numpy(dtype=float)  # feature matrix as an array
t = pd.Series(pd.to_datetime(utc, utc=True)).reset_index(drop=True)  # times
A = (pd.Timestamp('2019-09-26 18:25:29.940', tz='UTC'),  # window A start
     pd.Timestamp('2019-09-26 18:25:38.780', tz='UTC'))  # window A end
B = (pd.Timestamp('2019-09-26 18:27:20.185', tz='UTC'),  # window B start
     pd.Timestamp('2019-09-26 18:27:28.090', tz='UTC'))  # window B end


def intervals(flag):  # consecutive flagged rows, 2 or more
    run = flag.ne(flag.shift()).cumsum()  # number each run of equal flags
    g = pd.DataFrame({'t': t[flag], 'r': run[flag]})  # flagged rows only
    g = g.groupby('r')['t'].agg(['first', 'last', 'size'])  # per run
    return g[g['size'] > 1]  # keep runs of 2+ rows


def hits(iv, w):  # intervals that overlap the window w
    return int(((iv['first'] <= w[1]) & (iv['last'] >= w[0])).sum())


# contamination only moves the cut on the scores:
# cut = percentile(scores, 100 * c)
fits = {  # model settings: (trees, rows per tree)
    '100 trees, 256 rows': (100, 256),
    '300 trees, 256 rows': (300, 256),
    '300 trees, 1024 rows': (300, 1024),
}
rows, base_flag = [], {}  # result rows and the seed-0 reference flags
for name, (n, m) in fits.items():  # each model setting
    for seed in (0, 1, 2):  # each random seed
        s = IsolationForest(  # fit on all rows, then score every row
            n_estimators=n, max_samples=m, random_state=seed
        ).fit(values).score_samples(values)
        cuts = [('-0.5 default', -0.5)] + [  # default and percentile cuts
            (f'{c:.1%} lowest', np.percentile(s, 100 * c))
            for c in (0.005, 0.01, 0.02, 0.05)
        ]
        for label, cut in cuts:  # each cut
            flag = pd.Series(s < cut)  # rows below the cut
            iv = intervals(flag)  # their intervals
            key = (name, label)  # setting and cut
            if seed == 0:  # remember the seed-0 flags
                base_flag[key] = flag
            b = base_flag[key]  # seed-0 flags for the same setting and cut
            jac = (flag & b).sum() / max((flag | b).sum(), 1)  # overlap
            rows.append([name, label, seed, int(flag.sum()), len(iv),
                         hits(iv, A), hits(iv, B), round(jac, 2)])
out = pd.DataFrame(rows, columns=[  # results table
    'model', 'cut', 'seed', 'rows_flagged', 'intervals_2+rows',
    'hit_18:25 stretch', 'hit_18:27 burst', 'same_rows_as_seed0'])
print(out.to_string(index=False))  # show the table
print('\nsame_rows_as_seed0 = overlap of flagged rows with the seed-0 '
      'run (1.0 = identical).')
print('Candidates for investigation, not confirmed losses. '
      'Nothing here is tuned to reach a count.')

# export the tuning table to CSV
SAVE_PATH = OUT_DIR / 'b4_if_tuning_results.csv'  # output file
out.to_csv(SAVE_PATH, index=False)  # write the table
print(f'Saved: {SAVE_PATH}')


               model          cut  seed  rows_flagged  intervals_2+rows  hit_18:25 stretch  hit_18:27 burst  same_rows_as_seed0
 100 trees, 256 rows -0.5 default     0           437                23                  1                4                1.00
 100 trees, 256 rows  0.5% lowest     0            30                 3                  0                1                1.00
 100 trees, 256 rows  1.0% lowest     0            59                 7                  0                2                1.00
 100 trees, 256 rows  2.0% lowest     0           118                12                  1                5                1.00
 100 trees, 256 rows  5.0% lowest     0           298                28                  6                5                1.00
 100 trees, 256 rows -0.5 default     1           504                30                  2                3                0.47
 100 trees, 256 rows  0.5% lowest     1            29                 3                  0              

**What:**
- Fit Isolation Forests on X_minmax with three model settings (100 trees with 256 rows per tree, 300 trees with 256 rows, 300 trees with 1024 rows) and three random seeds each
- For each fit compare five cuts on the score: the library default -0.5 and the lowest 0.5%, 1%, 2% and 5% of the scores
- For each combination count the flagged rows, the intervals of 2 or more consecutive rows, how many of them touch two reference windows (18:25:29.940 to 18:25:38.780 and 18:27:20.185 to 18:27:28.090 UTC, both stretches flagged by the first seed-0 run), and the overlap of the flagged rows with the seed-0 run (same_rows_as_seed0)
- Save the table as b4_if_tuning_results.csv

**Why:**
- We want to know which setting changes the result and by how much, before we pick one
- The percentile cuts only move the line on the scores, they do not change the forest
- Nothing is tuned to reach a count of events

**Output:**
- 45 rows (3 settings x 3 seeds x 5 cuts)
- At the -0.5 default the number of flagged rows depends on the seed: 437, 504 and 538 for 100 trees, 357 to 539 for 300 trees, 256, 325 and 294 for 300 trees with 1024 rows
- The overlap with seed 0 at the -0.5 default is only 0.47 and 0.48 (100 trees), 0.69 and 0.59 (300 trees), 0.60 and 0.64 (1024 rows)
- The seed changes the flagged rows more than the number of trees or the rows per tree

**Next:** average the score over many seeds and check if that result is stable.

In [3]:
# Seed-averaged IF score: is the result stable once the seed is averaged out?
# Needs X_minmax and utc (same session). Takes about 1-3 minutes.
import numpy as np  # numerical tools
import pandas as pd  # tables
from sklearn.ensemble import IsolationForest  # the model

values = X_minmax.to_numpy(dtype=float)  # feature matrix as an array
t = pd.Series(pd.to_datetime(utc, utc=True)).reset_index(drop=True)  # times


def avg_score(seeds):  # mean score over one forest per seed
    return np.mean([
        IsolationForest(n_estimators=300, max_samples=256, random_state=s)
        .fit(values).score_samples(values) for s in seeds
    ], axis=0)


E1 = avg_score(range(0, 20))  # ensemble 1: 20 seeds
E2 = avg_score(range(100, 120))  # ensemble 2: 20 different seeds
S0 = IsolationForest(  # the old single run: 100 trees, seed 0
    n_estimators=100, max_samples=256, random_state=0
).fit(values).score_samples(values)
avg = (E1 + E2) / 2  # mean of all 40 forests


def n_intervals(flag):  # number of intervals of 2+ consecutive rows
    run = flag.ne(flag.shift()).cumsum()  # number each run
    sizes = pd.Series(flag.values)[flag.values].groupby(run[flag.values])
    return int((sizes.size() > 1).sum())  # runs with more than one row


rows = []  # one result row per cut
cuts = [('-0.5 default', -0.5, -0.5, -0.5)] + [  # default and percentile cuts
    (f'{c:.0%} lowest', np.percentile(E1, 100 * c),
     np.percentile(E2, 100 * c), np.percentile(S0, 100 * c))
    for c in (0.02, 0.05, 0.10)
]
for label, cut1, cut2, cut0 in cuts:  # each cut
    f1 = pd.Series(E1 < cut1)  # rows flagged by E1
    f2 = pd.Series(E2 < cut2)  # rows flagged by E2
    f0 = pd.Series(S0 < cut0)  # rows flagged by the old seed-0 run
    jac = (f1 & f2).sum() / max((f1 | f2).sum(), 1)  # E1 vs E2 overlap
    keep0 = (f0 & f1 & f2).sum() / max(f0.sum(), 1)  # old rows kept by both
    rows.append([label, int(f1.sum()), int(f2.sum()), round(jac, 2),
                 n_intervals(f1), n_intervals(f2), round(keep0, 2)])
out = pd.DataFrame(rows, columns=[  # results table
    'cut', 'rows_E1', 'rows_E2', 'E1_vs_E2_same_rows',
    'intervals_E1', 'intervals_E2', 'old_seed0_rows_kept_by_both'])
print(out.to_string(index=False))  # show the table
print(f'\nCorrelation of row scores, E1 vs E2: '
      f'{np.corrcoef(E1, E2)[0, 1]:.3f} | seed0 (100 trees) vs E1: '
      f'{np.corrcoef(S0, E1)[0, 1]:.3f}')
print('E1_vs_E2_same_rows = overlap of flagged rows between two '
      'independent 20-seed averages (1.0 = identical).')
score_path = OUT_DIR / 'b4_if_score_avg40.csv'
pd.DataFrame({'utc_time': t, 'score_avg40': avg}).to_csv(  # save scores
    score_path, index=False)
print(f'Saved: {score_path} (mean of 40 forests). '
      'Candidates only, not confirmed losses.')


         cut  rows_E1  rows_E2  E1_vs_E2_same_rows  intervals_E1  intervals_E2  old_seed0_rows_kept_by_both
-0.5 default      431      452                0.94            21            22                         0.68
   2% lowest      119      117                0.72            13            12                         0.69
   5% lowest      298      296                0.92            22            21                         0.66
  10% lowest      594      594                0.88            30            33                         0.72

Correlation of row scores, E1 vs E2: 0.999 | seed0 (100 trees) vs E1: 0.968
E1_vs_E2_same_rows = overlap of flagged rows between two independent 20-seed averages (1.0 = identical).
Saved: /Users/lakshminarayanareddymandu/Downloads/B4 Project/02_Data_and_Outputs/Generated_Outputs/b4_if_score_avg40.csv (mean of 40 forests). Candidates only, not confirmed losses.


**What:**
- Average the Isolation Forest score over many seeds: ensemble E1 (seeds 0 to 19) and ensemble E2 (seeds 100 to 119), each with 300 trees and 256 rows per tree, and the old single run S0 (100 trees, seed 0) for comparison
- Compare the rows flagged by E1 and E2 for the -0.5 default and for the lowest 2%, 5% and 10% of the scores
- Save the mean of the 40 forests as b4_if_score_avg40.csv (utc_time, score_avg40)

**Why:**
- Cell 2 showed that the seed changes the flagged rows more than any setting. If we average out the seed, two independent ensembles should agree with each other
- Takes about 1 to 3 minutes

**Output:**
- At -0.5: 431 rows in E1, 452 in E2, overlap 0.94, 21 and 22 intervals of 2+ rows. 68% of the old seed-0 rows are kept by both
- At the lowest 2%: 119 and 117 rows, overlap 0.72. At 5%: 298 and 296, overlap 0.92. At 10%: 594 and 594, overlap 0.88
- Correlation of the row scores, E1 vs E2: 0.999. Old seed 0 vs E1: 0.968
- So the averaged score is stable, while a single seed is not. The -0.5 default is the reference we keep

**Next:** list the intervals below -0.5 of the averaged score and look at the gaps between them.

In [4]:
# Intervals of the 40-forest averaged score at the -0.5 reference,
# plus gap analysis. Needs E1, E2 and utc from the previous cell.
import numpy as np  # numerical tools
import pandas as pd  # tables

t = pd.Series(pd.to_datetime(utc, utc=True)).reset_index(drop=True)  # times
avg = (E1 + E2) / 2  # mean score of the 40 forests
flag = pd.Series(avg < -0.5)  # rows below the reference
both = pd.Series((E1 < -0.5) & (E2 < -0.5))  # flagged by BOTH ensembles
run = flag.ne(flag.shift()).cumsum()  # number each run of equal flags

iv = (pd.DataFrame({'t': t[flag], 'r': run[flag],  # flagged rows only
                    'score': avg[flag], 'both': both[flag]})
      .groupby('r').agg(start=('t', 'first'), end=('t', 'last'),
                        rows=('t', 'size'), lowest=('score', 'min'),
                        both_share=('both', 'mean'))  # one row per interval
      .reset_index(drop=True))
iv['seconds'] = (iv['end'] - iv['start']).dt.total_seconds()  # length
multi = iv[iv['rows'] > 1].reset_index(drop=True)  # intervals of 2+ rows
print(f"Rows below -0.5 (averaged score): {int(flag.sum())} | "
      f"stretches: {len(iv)} ({len(multi)} of 2+ rows, "
      f"{int((iv['rows'] == 1).sum())} single rows)")
print(multi.assign(start=multi['start'].dt.strftime('%H:%M:%S.%f').str[:-3],
                   end=multi['end'].dt.strftime('%H:%M:%S.%f').str[:-3])
      .to_string(index=True, float_format=lambda x: f'{x:.3f}'))  # table

# Gap between consecutive 2+ row intervals (end of one to start of next).
gaps = (multi['start'].shift(-1) - multi['end']).dt.total_seconds().dropna()
print('\nGaps between consecutive intervals, seconds (sorted):')
print(np.round(np.sort(gaps.to_numpy()), 2).tolist())  # sorted gaps
print('\nEvents if intervals closer than X seconds are merged '
      '(no padding):')
for x in (0.25, 0.5, 1, 2, 3, 5, 10, 20):  # merge gaps to try
    print(f'  gap < {x:>5} s -> {int((gaps >= x).sum()) + 1} events')
print('\nLook for a plateau: a stretch of X values where the event '
      'count barely changes.')
print('Candidates only, not confirmed losses. '
      'Nothing here targets a count of 10.')

Rows below -0.5 (averaged score): 456 | stretches: 32 (22 of 2+ rows, 10 single rows)
           start           end  rows  lowest  both_share  seconds
0   18:21:16.045  18:21:17.915    23  -0.504       1.000    1.870
1   18:21:18.085  18:21:20.040    24  -0.506       1.000    1.955
2   18:21:46.305  18:21:47.495    15  -0.532       1.000    1.190
3   18:22:57.450  18:22:57.875     6  -0.503       1.000    0.425
4   18:22:58.385  18:22:58.895     7  -0.506       1.000    0.510
5   18:23:18.190  18:23:18.275     2  -0.502       0.500    0.085
6   18:23:20.910  18:23:21.165     4  -0.511       1.000    0.255
7   18:23:21.590  18:23:21.845     4  -0.508       1.000    0.255
8   18:23:22.015  18:23:24.055    25  -0.553       1.000    2.040
9   18:24:16.245  18:24:17.265    13  -0.515       0.846    1.020
10  18:24:17.435  18:24:18.030     8  -0.518       1.000    0.595
11  18:25:22.715  18:25:25.180    30  -0.542       1.000    2.465
12  18:25:31.980  18:25:35.890    47  -0.521       1.000

**What:**
- Take the 40-forest averaged score (avg), flag the rows below the -0.5 reference and group consecutive flagged rows into intervals
- For each interval report start, end, rows, lowest score, both_share (the share of its rows flagged by both E1 and E2) and the length in seconds
- Print the gaps between consecutive intervals of 2+ rows, and the number of events we get if intervals closer than a given gap are merged (no padding)

**Why:**
- To choose the merge gap from the data and not from a wanted event count. We look for a plateau where the event count barely changes
- Needs E1, E2 and utc from cell 3

**Output:**
- 456 rows below -0.5 in 32 stretches: 22 of 2+ rows and 10 single rows
- The 22 intervals run from 0.085 s (2 rows) to 5.865 s (70 rows). The lowest score is -0.574 (18:27:20.100 to 18:27:25.965). 21 of the 22 have both_share of at least 0.5 (the lowest of those is exactly 0.500), the interval at 18:27:48.575 has 0.000
- Sorted gaps in seconds: 0.17 (four times), 0.42, 0.51, 0.76, 2.04, 2.64, 6.8, 19.3 (twice), 20.57 and larger up to 69.96
- Events by merge gap: 18 (under 0.25 s), 17 (0.5 s), 15 (1 s and 2 s), 13 (3 s and 5 s), 12 (10 s), 10 (20 s)
- The event count stays at 13 for gaps between 3 s and 5 s, because the gaps jump from 2.64 s to 6.8 s

**Next:** build the 13 events with a 5 s gap and save them.

In [5]:
# Build the 13 events (gap 5 s) from the averaged score and save them
# for the hypothesis notebook. Needs E1, E2, utc from the earlier cells.
import numpy as np  # numerical tools
import pandas as pd  # tables

GAP = 5.0  # clean break between the 2.64 s and 6.8 s gaps
OUT = str(OUT_DIR) + '/'  # folder
t = pd.Series(pd.to_datetime(utc, utc=True)).reset_index(drop=True)  # times
avg = (E1 + E2) / 2  # mean score of the 40 forests
flag = pd.Series(avg < -0.5)  # rows below the reference
both = pd.Series((E1 < -0.5) & (E2 < -0.5))  # flagged by BOTH ensembles
run = flag.ne(flag.shift()).cumsum()  # number each run of equal flags
iv = (pd.DataFrame({'t': t[flag], 'r': run[flag],  # flagged rows only
                    'score': avg[flag], 'both': both[flag]})
      .groupby('r').agg(start=('t', 'first'), end=('t', 'last'),
                        rows=('t', 'size'), lowest=('score', 'min'),
                        both_share=('both', 'mean'))  # one row per interval
      .reset_index(drop=True))
iv = iv[iv['rows'] > 1].reset_index(drop=True)  # keep intervals of 2+ rows
gap_s = (iv['start'] - iv['end'].shift()).dt.total_seconds()  # gap before
iv['event'] = (gap_s >= GAP).cumsum() + 1  # new event after a 5 s gap
ev = (iv.groupby('event').agg(  # one row per event
    start_utc=('start', 'min'), end_utc=('end', 'max'),
    flagged_rows=('rows', 'sum'), intervals=('rows', 'size'),
    lowest_score=('lowest', 'min'), both_share=('both_share', 'mean'))
      .reset_index())
ev['both_share'] = ev['both_share'].round(3)  # round for display
ev['lowest_score'] = ev['lowest_score'].round(4)
ev['seconds'] = (ev['end_utc'] - ev['start_utc']
                 ).dt.total_seconds().round(3)  # event length
ev.to_csv(OUT + 'b4_if_events_gap5.csv', index=False)  # save the events
iv.to_csv(OUT + 'b4_if_intervals_avg.csv', index=False)  # and intervals

print(f'{len(iv)} intervals (2+ rows) -> {len(ev)} events, '
      f'merged when gap < {GAP} s, no padding.')
show = ev.assign(  # readable times for printing
    start_utc=ev['start_utc'].dt.strftime('%H:%M:%S.%f').str[:-3],
    end_utc=ev['end_utc'].dt.strftime('%H:%M:%S.%f').str[:-3])
print(show.to_string(index=False))  # event table
print(f'\nSaved: b4_if_events_gap5.csv and b4_if_intervals_avg.csv '
      f'in {OUT}')
print('Source of the events: averaged score at -0.5. '
      'Candidates only, not confirmed losses.')


22 intervals (2+ rows) -> 13 events, merged when gap < 5.0 s, no padding.
 event    start_utc      end_utc  flagged_rows  intervals  lowest_score  both_share  seconds
     1 18:21:16.045 18:21:20.040            47          2       -0.5059       1.000    3.995
     2 18:21:46.305 18:21:47.495            15          1       -0.5320       1.000    1.190
     3 18:22:57.450 18:22:58.895            13          2       -0.5064       1.000    1.445
     4 18:23:18.190 18:23:24.055            35          4       -0.5530       0.875    5.865
     5 18:24:16.245 18:24:18.030            21          2       -0.5177       0.923    1.785
     6 18:25:22.715 18:25:25.180            30          1       -0.5416       1.000    2.465
     7 18:25:31.980 18:25:38.695            57          2       -0.5212       0.950    6.715
     8 18:26:17.965 18:26:18.050             2          1       -0.5033       1.000    0.085
     9 18:26:58.170 18:26:59.530            16          2       -0.5183       1.000    1.

**What:**
- Build the events from the averaged score: flag rows below -0.5, group them into intervals of 2+ rows, and start a new event when the gap to the previous interval is 5 s or more (GAP = 5.0, no padding)
- For each event report start, end, flagged rows, number of intervals, lowest score, mean both_share and length in seconds
- Save the events as b4_if_events_gap5.csv and the intervals as b4_if_intervals_avg.csv for the hypothesis notebook

**Why:**
- The gap table in cell 4 jumps from 2.64 s to 6.8 s, so 5 s separates the two groups of gaps cleanly. We do not choose it to reach a count
- Needs E1, E2 and utc from the earlier cells

**Output:**
- 22 intervals (2+ rows) become 13 events (UTC start times: 18:21:16.045, 18:21:46.305, 18:22:57.450, 18:23:18.190, 18:24:16.245, 18:25:22.715, 18:25:31.980, 18:26:17.965, 18:26:58.170, 18:27:20.100, 18:27:48.575, 18:28:23.935, 18:28:49.690)
- Lowest score and most flagged rows: event 10 (-0.5742, 70 rows, 5.865 s). Longest in time: event 7 (6.715 s, 57 rows)
- Shortest: event 8 (2 rows, 0.085 s). Event 2 has 15 rows in 1.19 s
- Event 11 (18:27:48.575 to 18:27:50.530, 24 rows) has both_share 0.000: the averaged score flags it, but neither ensemble alone does
- Events 4, 5, 7 and 12 are merged from 2 to 4 intervals each (event 4 from 4)

**Next:** cells 6 to 9 are the older seed-0 baseline.
